In [10]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable
import torch.nn.init as weight_init
import sys  
from rich import print as rprint

In [32]:
import sys
import os
from pathlib import Path

# Get the current working directory
current_dir = Path.cwd()

# Find the src directory (parent of autoencoder or search up)
src_dir = None
if current_dir.name == 'autoencoder':
    src_dir = current_dir.parent
else:
    # Search up the directory tree to find src
    for parent in [current_dir] + list(current_dir.parents):
        if (parent / 'src' / 'models-architecture').exists():
            src_dir = parent / 'src'
            break

if src_dir:
    models_path = str((src_dir / 'models-architecture').resolve())
    utils_path = str((src_dir / 'utils').resolve())
    
    sys.path.insert(0, models_path)
    sys.path.insert(0, utils_path)
    
    rprint(f"Added to path: {models_path}")
    rprint(f"Added to path: {utils_path}")
else:
    rprint("Warning: Could not find src directory")

from autoencoder import Encoder
import preprocessor

Added to path: /Users/abbas/Documents/Codes/thesis/movie-mate/src/models-architecture

Added to path: /Users/abbas/Documents/Codes/thesis/movie-mate/src/utils

In [33]:
from sklearn.model_selection import train_test_split

In [34]:
ratings_df = pd.read_csv('../../data/mv-lens-1m/ratings.dat', delimiter = '::',header=None, engine='python')

In [35]:
ratings_df.head()

,0,1,2,3
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [36]:
ratings_df.columns = ['userid','itemid','rating','timestamp']


In [27]:
ratings_df.head()

,userid,itemid,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [37]:
ratings_reindex = preprocessor.reindexer(ratings_df,'userid','itemid','rating')


In [38]:
train, test = train_test_split(ratings_reindex,
                               stratify=ratings_reindex['encoded_users'],
                               test_size=0.1,
                               random_state=42)

In [39]:
print(train)

        encoded_users  encoded_items  rating_col
141493            911           3441           3
199653           1224           3489           5
301642           1792           1827           5
621687           3766           2110           3
846029           5084             48           2
...               ...            ...         ...
446781           2753           3217           3
898456           5433           1913           1
862268           5186           1179           5
107551            709           1825           5
785615           4694           3237           5

[900188 rows x 3 columns]


In [40]:
print(test)

        encoded_users  encoded_items  rating_col
180052           1127           2748           3
666617           4013           1479           4
843034           5068           1141           4
6077               44            371           3
958086           5782           2231           4
...               ...            ...         ...
72344             482            973           3
98765             660            444           3
65909             442            854           5
635807           3833           3042           5
181184           1132           2737           4

[100021 rows x 3 columns]


In [49]:
full_records = np.array(ratings_reindex, dtype = 'int')
full_records = torch.FloatTensor(full_records)

training_set = np.array(train, dtype = 'int')
test_set = np.array(test, dtype = 'int')

nb_users = int(max(max(training_set[:,0]), max(test_set[:,0])))
nb_movies = int(max(max(training_set[:,1]), max(test_set[:,1])))

rprint(f"Total Number of Users: {nb_users}")
rprint(f"Total number of Movies: {nb_movies}")

Total Number of Users: 6040

Total number of Movies: 3706

In [50]:
def convert(data):
    new_data = []
    for id_users in range(nb_users+1):
        # each user's watched movies
        # data[:,0], first column, all rows column users
        id_items = data[:,1][data[:,0] == id_users]
        # each user's rating for that item
        id_ratings = data[:,2][data[:,0] == id_users]
        ratings = np.zeros(nb_movies)
        # the positions of these items are filled with ratings, creating the matrix
        ratings[id_items-1] = id_ratings
        new_data.append(list(ratings))
    return new_data
  
training_set = convert(training_set)
test_set = convert(test_set)

training_set = torch.FloatTensor(training_set)
test_set = torch.FloatTensor(test_set)

In [51]:
autoencoder_network = Encoder([nb_movies,20,10],'sigmoid',0.1)
criterion = nn.MSELoss()
optimizer = optim.RMSprop(autoencoder_network.parameters(), lr = 0.01, weight_decay = 0.5)

nb_epoch = 10
for epoch in range(1, nb_epoch + 1):
    train_loss = 0
    s = 0.
    # s is the number of users who rated at least 1 movies
    for id_user in range(nb_users):
        input = Variable(training_set[id_user]).unsqueeze(0)
        target = input.clone()
        if torch.sum(target.data > 0) > 0:
            output = autoencoder_network(input)
            target.require_grad = False
            output[target == 0] = 0
            loss = criterion(output, target)
            mean_corrector = nb_movies/float(torch.sum(target.data > 0) + 1e-10) #making this anyway not equal to 0, as this will be a denominator
            #mean_corrector is the avg of the error, only considering the movies having ratings (non-zero ratings) for computing mean of error
            loss.backward() # decide the direction the increment of weights
            #this call will just computing all the gradients required
            train_loss += np.sqrt(loss.data*mean_corrector)
            s += 1.
            optimizer.step() # decide the amount to update the weights
            
    print('epoch: '+str(epoch)+' loss: '+ str(train_loss/s))

/var/folders/17/wfrxtzsx43b7gf_s8lyk_yjw0000gn/T/ipykernel_32296/3897330205.py:22: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  train_loss += np.sqrt(loss.data*mean_corrector)


epoch: 1 loss: tensor(1.2799)
epoch: 2 loss: tensor(0.9991)
epoch: 3 loss: tensor(0.9853)
epoch: 4 loss: tensor(0.9804)
epoch: 5 loss: tensor(0.9786)
epoch: 6 loss: tensor(0.9770)
epoch: 7 loss: tensor(0.9767)
epoch: 8 loss: tensor(0.9758)
epoch: 9 loss: tensor(0.9755)
epoch: 10 loss: tensor(0.9750)


In [52]:
test_loss = 0
s = 0.

res = []
targets = []

# averaged difference between real rating and predicted rating
for id_user in range(nb_users):
    input = Variable(training_set[id_user]).unsqueeze(0) # should keep the training set
    target = Variable(test_set[id_user]).unsqueeze(0) # to predict the other movies user not seen yet
    
    if torch.sum(target.data > 0) > 0:
        # make predictions
        output = autoencoder_network(input)
        targets.append(target.detach().numpy())
        res.append(output.detach().numpy()) 
        target.require_grad = False
        output[target == 0] = 0 # dont want to measure the loss on the movies didnt get the actual rating from user 
        # force to 0 and difference / loss will be 0 for those entries
        loss = criterion(output, target)
        
        mean_corrector = nb_movies/float(torch.sum(target.data > 0) + 1e-10) 
        # only consider the movies that are rated in the test set, to be included in the loss
        test_loss += np.sqrt(loss.data*mean_corrector)
        s += 1.
print('test loss: '+str(test_loss/s))

/var/folders/17/wfrxtzsx43b7gf_s8lyk_yjw0000gn/T/ipykernel_32296/1643896338.py:24: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  test_loss += np.sqrt(loss.data*mean_corrector)


test loss: tensor(0.9523)


In [53]:
def make_top_k_recommendations(encoder,evidence,k,filter_seen=True):
    '''
    :param encoder: autoencoder instance
    :param evidence: full set of seen ratings from all users
    :param k: top k items (by output score)
    :param filter_seen: (default True) filter controller to remove seen items from top k list
    '''     
    res = []
    nb_users = evidence.shape[0]
    # to find top scored items for each user
    for id_user in range(nb_users):
        encoder_input = Variable(evidence[id_user]).unsqueeze(0) # should keep the training set 
        encoder_output = encoder(encoder_input)
        
        target = Variable(evidence[id_user]).unsqueeze(0) # mask to find items not seen yet
        if filter_seen:
            encoder_output[target != 0] = 0 # force seen items scores to 0, will never get recommended
        res.append(encoder_output.detach().numpy())
        
    res = [a[0] for a in res]
    final_itemsets = []    
    for each in res:
        full_ratings_predicted = list(each)
        full_ratings_indexed = list(enumerate(full_ratings_predicted))
        final_itemsets.append(sorted(full_ratings_indexed,key=lambda x:x[1],reverse =True)[:k])
        
    return final_itemsets